# Experiment 4: Cliff Walking Sensitivity Analysis

Phân tích độ nhạy của SARSA và Q-Learning theo learning rate và exploration schedule dưới cùng environment-step budget.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPOSITORY = 'https://github.com/Briannguyen3106/project1.git'
ROOT = Path('/kaggle/working/project1')
if ROOT.exists():
    subprocess.run(['git', '-C', str(ROOT), 'pull', '--ff-only'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY, str(ROOT)], check=True)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import src.Environment4 as experiment
experiment.OUTPUT_DIR = Path('/kaggle/working/results/figures/environment4_cliff_sensitivity')
run_experiment = experiment.run_experiment
OUTPUT_DIR = experiment.OUTPUT_DIR
print(f'Project root: {ROOT}')
print(f'Output directory: {OUTPUT_DIR}')

In [ ]:
%cd /kaggle/working/project1
!pip install -r requirements.txt

## Thiết kế

- Alpha sweep: `{0.01, 0.1, 0.3, 0.5, 0.8}`, giữ epsilon = 0.1 cố định.
- Schedule sweep: epsilon 0.1 cố định so với decay 0.1 -> 0.01, giữ alpha = 0.1.
- 30 seed, 30.000 training transitions cho mọi run.
- Exact greedy evaluation mỗi 1.200 steps.
- Threshold -20 phải duy trì 3 checkpoint liên tiếp.

In [ ]:
episodes_df, evaluations_df, seed_metrics = run_experiment(
    seeds=list(range(30)),
    step_budget=30_000,
)
print(f'Results saved to: {OUTPUT_DIR}')

## Sanity check

Mọi condition phải có đúng cùng step budget trên từng seed.

In [ ]:
step_check = (
    episodes_df.groupby(['experiment', 'condition', 'algorithm', 'seed'])['cumulative_steps']
    .max().groupby(['experiment', 'condition', 'algorithm']).agg(['min', 'max'])
)
display(step_check)

## Learning-rate sensitivity

Báo cáo final greedy return, cliff-fall rate cuối, tỷ lệ seed đạt threshold và AUC.

In [ ]:
final_eval = (
    evaluations_df.sort_values('cumulative_steps')
    .groupby(['experiment', 'condition', 'algorithm', 'seed'], as_index=False).tail(1)
)
late_training = episodes_df[episodes_df['cumulative_steps'] > 24_000]
alpha_final = final_eval[final_eval['experiment'] == 'alpha'].groupby(
    ['condition', 'algorithm']
)[['eval_return', 'eval_success_rate']].agg(['mean', 'std', 'median'])
alpha_safety = late_training[late_training['experiment'] == 'alpha'].groupby(
    ['condition', 'algorithm']
)[['return', 'cliff_falls']].mean()
alpha_efficiency = seed_metrics[seed_metrics['experiment'] == 'alpha'].groupby(
    ['condition', 'algorithm']
).agg(
    threshold_reach_rate=('reached_threshold', 'mean'),
    steps_to_threshold=('steps_to_threshold', 'mean'),
    return_auc=('normalized_return_auc', 'mean'),
)
display(alpha_final)
display(alpha_safety)
display(alpha_efficiency)

## Exploration-schedule sensitivity

So sánh fixed và decay trong từng thuật toán; không diễn giải lịch decay đến 0.01 là GLIE.

In [ ]:
epsilon_final = final_eval[final_eval['experiment'] == 'epsilon_schedule'].groupby(
    ['condition', 'algorithm']
)[['eval_return', 'eval_success_rate']].agg(['mean', 'std', 'median'])
epsilon_safety = late_training[late_training['experiment'] == 'epsilon_schedule'].groupby(
    ['condition', 'algorithm']
)[['return', 'cliff_falls']].mean()
epsilon_efficiency = seed_metrics[seed_metrics['experiment'] == 'epsilon_schedule'].groupby(
    ['condition', 'algorithm']
).agg(
    threshold_reach_rate=('reached_threshold', 'mean'),
    steps_to_threshold=('steps_to_threshold', 'mean'),
    return_auc=('normalized_return_auc', 'mean'),
)
display(epsilon_final)
display(epsilon_safety)
display(epsilon_efficiency)

In [ ]:
import shutil
archive = shutil.make_archive('/kaggle/working/environment4_cliff_sensitivity', 'zip', OUTPUT_DIR)
print(f'Archive ready: {archive}')